In [1]:
import asyncio
import csv
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
import nest_asyncio
from pathlib import Path
import ast

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Import your existing conversation generator
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator
from prompts import TERMINATION_SIGNAL

class MovieConversationGenerator:
    """Generator for movie recommendation conversations based on CSV data"""
    
    def __init__(self, config: ConversationConfig, prompt_template_path: str, terminal_signal: str = "[[TERMINATE CHAT]]"):
        self.config = config
        self.terminal_signal = terminal_signal
        self.generator = MultiTurnConversationGenerator(config)
        
        # Load prompt template
        with open(prompt_template_path, 'r') as f:
            self.prompt_template = f.read()
    
    def load_csv_data(self, csv_path: str) -> pd.DataFrame:
        """Load and parse the CSV data"""
        df = pd.read_csv(csv_path)
        
        # Parse the conversation column (assuming it's stored as string representation of list)
        def parse_conversation(conv_str):
            try:
                # Handle the conversation string - it might be a JSON string or Python literal
                if isinstance(conv_str, str):
                    return ast.literal_eval(conv_str)
                return conv_str
            except (ValueError, SyntaxError) as e:
                logger.warning(f"Failed to parse conversation: {e}")
                return []
        
        df['conversation_parsed'] = df['conversation'].apply(parse_conversation)
        return df
    
    def create_user_prompt(self, conversation: List[Dict], ground_truth: str) -> str:
        """Create user prompt from conversation and ground truth"""
        # Convert conversation to a readable format
        conv_text = ""
        for msg in conversation:
            role = msg['role']
            content = msg['content']
            if role == 'user':
                conv_text += f"User: {content}\n"
            elif role == 'assistant':
                conv_text += f"Assistant: {content}\n"
        
        # Fill in the template
        prompt = self.prompt_template.format(
            conversation=conv_text.strip(),
            ground_truth=ground_truth,
            terminal_signal=self.terminal_signal
        )
        
        return prompt
    
    async def generate_single_movie_conversation(self, dialog_id: str, conversation: List[Dict], ground_truth: str) -> Optional[Dict]:
        """Generate a single conversation based on the movie data"""
        try:
            # Create the user prompt
            user_prompt = self.create_user_prompt(conversation, ground_truth)
            print(user_prompt)
            
            logger.info(f"Generating conversation for dialog_id: {dialog_id}")
            
            # Generate the conversation
            generated_conv = await self.generator.generate_single_conversation(user_prompt)
            
            if generated_conv:
                result = {
                    'dialog_id': dialog_id,
                    'ground_truth': ground_truth,
                    'original_conversation': conversation,
                    'generated_conversation': generated_conv,
                    'status': 'success'
                }
                logger.info(f"Successfully generated conversation for {dialog_id}")
                return result
            else:
                logger.warning(f"Failed to generate conversation for {dialog_id}")
                return {
                    'dialog_id': dialog_id,
                    'ground_truth': ground_truth,
                    'original_conversation': conversation,
                    'generated_conversation': None,
                    'status': 'failed'
                }
                
        except Exception as e:
            logger.error(f"Error generating conversation for {dialog_id}: {str(e)}")
            return {
                'dialog_id': dialog_id,
                'ground_truth': ground_truth,
                'original_conversation': conversation,
                'generated_conversation': None,
                'status': 'error',
                'error': str(e)
            }
    
    async def generate_conversations_batch(self, df: pd.DataFrame, batch_size: Optional[int] = None) -> List[Dict]:
        """Generate conversations for all rows in the dataframe"""
        if batch_size is None:
            batch_size = self.config.batch_size
        
        total_rows = len(df)
        logger.info(f"Starting batch generation for {total_rows} conversations")
        
        # Create tasks for all conversations
        tasks = []
        for idx, row in df.iterrows():
            task = self.generate_single_movie_conversation(
                dialog_id=row['dialog_id'],
                conversation=row['conversation_parsed'],
                ground_truth=row['ground_truth']
            )
            tasks.append(task)
        
        # Process in batches to avoid overwhelming the system
        results = []
        for i in range(0, len(tasks), batch_size):
            batch_tasks = tasks[i:i + batch_size]
            batch_num = i // batch_size + 1
            total_batches = (len(tasks) + batch_size - 1) // batch_size
            
            logger.info(f"Processing batch {batch_num}/{total_batches} ({len(batch_tasks)} conversations)")
            
            start_time = time.time()
            batch_results = await asyncio.gather(*batch_tasks, return_exceptions=True)
            end_time = time.time()
            
            # Handle any exceptions in the batch
            for j, result in enumerate(batch_results):
                if isinstance(result, Exception):
                    logger.error(f"Exception in batch {batch_num}, item {j}: {result}")
                    # Create error result
                    row_idx = i + j
                    row = df.iloc[row_idx]
                    error_result = {
                        'dialog_id': row['dialog_id'],
                        'ground_truth': row['ground_truth'],
                        'original_conversation': row['conversation_parsed'],
                        'generated_conversation': None,
                        'status': 'exception',
                        'error': str(result)
                    }
                    results.append(error_result)
                else:
                    results.append(result)
            
            logger.info(f"Batch {batch_num} completed in {end_time - start_time:.2f} seconds")
        
        successful = sum(1 for r in results if r['status'] == 'success')
        logger.info(f"Generation complete: {successful}/{total_rows} successful")
        
        return results
    
    def save_results_to_csv(self, results: List[Dict], output_path: str):
        """Save results to CSV file"""
        # Prepare data for CSV
        csv_data = []
        for result in results:
            # Convert generated conversation to string for CSV storage
            generated_conv_str = json.dumps(result['generated_conversation']) if result['generated_conversation'] else None
            original_conv_str = json.dumps(result['original_conversation'])
            
            csv_row = {
                'dialog_id': result['dialog_id'],
                'ground_truth': result['ground_truth'],
                'original_conversation': original_conv_str,
                'generated_conversation': generated_conv_str,
                'status': result['status']
            }
            
            # Add error information if present
            if 'error' in result:
                csv_row['error'] = result['error']
            
            csv_data.append(csv_row)
        
        # Write to CSV
        df_output = pd.DataFrame(csv_data)
        df_output.to_csv(output_path, index=False)
        logger.info(f"Results saved to {output_path}")
        
        # Print summary
        status_counts = df_output['status'].value_counts()
        logger.info(f"Summary: {status_counts.to_dict()}")

# Configuration for movie conversation generation
def create_movie_config():
    """Create configuration for movie recommendation conversations"""
    return ConversationConfig(
        task_desc="You are a helpful movie recommendation assistant. Provide personalized movie suggestions based on user preferences and engage in natural conversation about movies.",
        max_total_turns=12,  # Allow longer conversations for movie recommendations
        max_gen_workers=4,   # Adjust based on your system
        local_model_path="/home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test",
        base_model_path="meta-llama/Llama-3.2-1B-Instruct",
        assistant_generation_kwargs={
            "temperature": 0.7,
            "max_tokens": 512
        },
        user_generation_kwargs={
            "model": "anthropic.claude-3-sonnet-20240229-v1:0",
            "temperature": 0.8,
            "max_tokens": 256,
            "num_retries": 5
        },
        batch_size=3,  # Smaller batch size for stability
        enable_batching=True
    )

# Main execution function
async def main():
    """Main function to run the movie conversation generation"""
    
    # File paths
    csv_input_path = "../datasets/inspired/multiturn_form/test.csv"
    prompt_template_path = "../prompts/test_user_prompt.txt"
    csv_output_path = "multiturn_test/llama3_2_1B/inspired/generated_movie_conversations.csv"
    
    # Create configuration
    config = create_movie_config()
    
    # Initialize generator
    movie_generator = MovieConversationGenerator(
        config=config,
        prompt_template_path=prompt_template_path,
        terminal_signal=TERMINATION_SIGNAL
    )
    
    # Load data
    logger.info("Loading CSV data...")
    df = movie_generator.load_csv_data(csv_input_path)
    logger.info(f"Loaded {len(df)} conversations from CSV")
    
    # Generate conversations
    logger.info("Starting conversation generation...")
    start_time = time.time()
    
    results = await movie_generator.generate_conversations_batch(df)
    
    end_time = time.time()
    logger.info(f"Total generation time: {end_time - start_time:.2f} seconds")
    
    # Save results
    logger.info("Saving results...")
    movie_generator.save_results_to_csv(results, csv_output_path)
    
    logger.info("Process complete!")
    
    return results

INFO 07-09 20:55:00 [__init__.py:244] Automatically detected platform cuda.


In [3]:
# async def test_single_conversation():
#     """Test with a single conversation"""
#     config = create_movie_config()
#     generator = MovieConversationGenerator(
#         config=config,
#         prompt_template_path="../prompts/test_user_prompt.txt",
#         terminal_signal="[[TERMINATE CHAT]]"
#     )
    
#     # Test data (based on your example)
#     test_conversation = [
#         {'role': 'assistant', 'content': "Hi! I'm here to help you chose a movie!"},
#         {'role': 'user', 'content': 'Terrific'},
#         {'role': 'assistant', 'content': 'What are some genres you like? What was the last movie you saw?'},
#         {'role': 'user', 'content': 'the last movie i saw in the theater was "Hustlers". I generally like comedy, drama and documentaries'},
#         # ... (truncated for brevity)
#     ]
    
#     result = await generator.generate_single_movie_conversation(
#         dialog_id="test_001",
#         conversation=test_conversation,
#         ground_truth="A Beautiful Day in the Neighborhood"
#     )
    
#     print("Test result:", result['status'])
#     if result['generated_conversation']:
#         print(f"Generated {len(result['generated_conversation'])} messages")
#         for msg in result['generated_conversation']:
#             print(f"{msg['role']}: {msg['content'][:100]}...")

# # Uncomment to run test
# await test_single_conversation()

# Run main process
results = await main()

🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-09 20:55:29 [config.py:823] This model supports multiple tasks: {'reward', 'score', 'generate', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 07-09 20:55:29 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-09 20:55:29 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-09 20:55:30 [core.py:455] Waiting for init message from front-end.
INFO 07-09 20:55:30 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=2 pid=150751) INFO 07-09 20:55:35 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=2 pid=150751) INFO 07-09 20:55:35 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=4 pid=150756) INFO 07-09 20:55:35 [default_loader.py:272] Loading weights took 0.11 seconds
(VllmWorker rank=3 pid=150755) INFO 07-09 20:55:35 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=4 pid=150756) INFO 07-09 20:55:35 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=6 pid=150758) INFO 07-09 20:55:35 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=150749) INFO 07-09 20:55:35 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=0 pid=150749) INFO 07-09 20:55:35 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=3 pid=150755) INFO 07-09 20:55:35 [default_loader.py:272] Loading weights took 0.10 seconds
(VllmWorker rank=7 pid=

INFO: ✅ Successfully initialized vLLM with LoRA: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO: Loading CSV data...
INFO: Loaded 99 conversations from CSV
INFO: Starting conversation generation...
INFO: Starting batch generation for 99 conversations
INFO: Processing batch 1/33 (3 conversations)
INFO: Generating conversation for dialog_id: 20191127-224739_530_live.pkl
INFO: Generating conversation for dialog_id: 20191130-081727_440_live.pkl
INFO: Generating conversation for dialog_id: 20191201-175152_742_live.pkl


✅ Conversation generator ready!
Below is a conversation between a user and a movie recommendation assistant. 

### Conversation:
Assistant: Hi! I'm here to help you chose a movie!
User: Terrific
Assistant: What are some genres you like? What was the last movie you saw?
User: the last movie i saw in the theater was QUOTATION_MARKHustlersQUOTATION_MARK . I generally like comedy, drama and documentaries
Assistant: How did you like QUOTATION_MARKHustlersQUOTATION_MARK? It definitely has the drama aspect, did it leave you wanting more or was it not exactly what you were looking for?
User: I liked it, it wasn't the most high-brow movie i've ever seen but it was fun. I like to just enjoy
Assistant: And what are your thoughts on action in film?
User: I like some action. Just not too violent
Assistant: Definitely, movies like QUOTATION_MARKDeadpoolQUOTATION_MARK wouldn't be up your alley I'm guessing?
User: It could be. I understand Deadpool was very funny I like funny
Assistant: Haha it defini

...

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

:
Traceback (most recent call last):


Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 667, in process_input_sockets
    request_type = EngineCoreRequestType(
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/enum.py", line 385, in __call__
    return cls.__new__(cls, value)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/enum.py", line 710, in __new__
    raise ve_exc
ValueError: b'\x00\x00' is not a valid EngineCoreRequestType


In [6]:
import os
path_to_check = "../prompts/test_user_prompt.txt" # Example path

if os.path.exists(path_to_check):
    print(f"The path '{path_to_check}' exists.")
else:
    print(f"The path '{path_to_check}' does not exist.")

The path '../prompts/test_user_prompt.txt' exists.
